In [9]:
import os, json, random, math, time
from collections import defaultdict
from tqdm import tqdm

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, average_precision_score

In [10]:
GRAPH_DIR =r"C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/data/processed_graph"
OUT_DIR = os.path.join(GRAPH_DIR, "pipeline_outputs")
os.makedirs(OUT_DIR, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEED = 42
set_seed = lambda s=SEED: (random.seed(s), np.random.seed(s), torch.manual_seed(s), torch.cuda.manual_seed_all(s) if torch.cuda.is_available() else None)
set_seed(SEED)
# Hyperparameters
HDIM = 128
OUTDIM = 128
EPOCHS = 6
BATCH_SIZE = 2048
NEIGHBOR_SAMPLES = [20,10]  # unused in CPU-only
LR = 1e-3
WEIGHT_DECAY = 1e-6
PATH_REG_WEIGHT = 0.5
NEG_RATIO = 1
ENSEMBLE_SIZE = 2
MC_RUNS = 30
TOP_K = 50

In [11]:


# -------------------- HELPER FUNCTIONS --------------------
def safe_read_lines(path):
    if not os.path.exists(path): return []
    with open(path,'r',encoding='utf-8',errors='ignore') as f:
        return [ln.rstrip("\n") for ln in f]

def load_edge_files(graph_dir):
    ei = os.path.join(graph_dir,"edge_index.pt")
    et = os.path.join(graph_dir,"edge_type.pt")
    assert os.path.exists(ei) and os.path.exists(et)
    edge_index = torch.load(ei).long()
    edge_type = torch.load(et).long()
    return edge_index, edge_type

def read_inductive_csv(path):
    if not os.path.exists(path): return []
    df = pd.read_csv(path, header=None, dtype=str)
    if df.shape[0] > 0 and [c.lower() for c in df.iloc[0].astype(str).tolist()] == ['head','relation','tail']:
        df = df.iloc[1:].reset_index(drop=True)
    triples = [(str(r[0]).strip(), str(r[1]).strip(), str(r[2]).strip()) for r in df.values if len(r)>=3]
    return triples



In [12]:
# -------------------- LOAD FILES --------------------
entities_lines = safe_read_lines(os.path.join(GRAPH_DIR,"C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/data/processed_graph/edge_index.pt"))
relation_lines = safe_read_lines(os.path.join(GRAPH_DIR,"C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/data/processed_graph/relations.txt"))
edge_index, edge_type = load_edge_files(GRAPH_DIR)

train_triples = read_inductive_csv(os.path.join(GRAPH_DIR,"C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/data/train_inductive.csv"))
val_triples   = read_inductive_csv(os.path.join(GRAPH_DIR,"C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/data/val_inductive.csv"))
test_triples  = read_inductive_csv(os.path.join(GRAPH_DIR,"C:/Users/Manasa/OneDrive/Desktop/Drug_Repurposing_Gnn/Drug_Repurposing_Gnn/data/test_inductive.csv"))

print("Loaded entities:", len(entities_lines), "relations:", len(relation_lines))
print("Edge index shape:", tuple(edge_index.shape), "edge_type len:", len(edge_type))
print("Inductive triples train/val/test:", len(train_triples), len(val_triples), len(test_triples))



Loaded entities: 221307 relations: 107
Edge index shape: (2, 5261827) edge_type len: 5261827
Inductive triples train/val/test: 5261827 584648 27786


In [13]:
# -------------------- PARSE ENTITIES --------------------
ent2name = {}; ent2type = {}
for ln in entities_lines:
    parts = ln.split("\t")
    if len(parts) >= 2:
        gid_str = parts[0].strip()
        label = parts[1].strip()
        ent2name[gid_str] = label
        lab_low = label.lower()
        if "compound::" in lab_low or "drug" in lab_low or "atc::" in lab_low:
            typ = "compound"
        elif "gene::" in lab_low:
            typ = "gene"
        elif "disease::" in lab_low or "doid:" in lab_low:
            typ = "disease"
        else:
            typ = "other"
        ent2type[gid_str] = typ
    else:
        idx = str(len(ent2name))
        ent2name[idx] = ln.strip()
        ent2type[idx] = "other"

label_to_gid = {v:k for k,v in ent2name.items()}
gidstr_to_int = {}
for gid_str in ent2name.keys():
    try: gidstr_to_int[gid_str] = int(gid_str)
    except: pass

num_nodes = int(edge_index.max().item()) + 1
print("num_nodes inferred from edge_index:", num_nodes)


num_nodes inferred from edge_index: 94046


In [14]:

# -------------------- PARSE RELATIONS --------------------
rid_to_relname = {}
for ln in relation_lines:
    parts = ln.split("\t")
    if len(parts) >= 2:
        rid_to_relname[parts[0].strip()] = parts[1].strip()

pattern_cg = ["compound:gene","target","bind","hetionet::cbg","dgidb::","gnbr::a","gnbr::b","gnbr::e"]
pattern_gd = ["gene:disease","assoc","associated","gnbr::d","gnbr::g","gnbr::md","gnbr::mp","gnbr::j"]

def relname_indicates(relname, patterns):
    if relname is None: return False
    rl = relname.lower()
    return any(p in rl for p in patterns)

def map_triple_label_to_int(triple):
    h,r,t = triple
    if h in label_to_gid and t in label_to_gid:
        try: return int(label_to_gid[h]), r, int(label_to_gid[t])
        except: return None
    try:
        hi,ti = int(h), int(t)
        if 0 <= hi < num_nodes and 0 <= ti < num_nodes: return hi,r,ti
    except: pass
    return None

train_int = [map_triple_label_to_int(t) for t in train_triples]; train_int = [t for t in train_int if t is not None]
val_int   = [map_triple_label_to_int(t) for t in val_triples]; val_int = [t for t in val_int if t is not None]
test_int  = [map_triple_label_to_int(t) for t in test_triples]; test_int = [t for t in test_int if t is not None]



In [15]:
# -------------------- EXTRACT MECHANISTIC PAIRS --------------------
comp_gene_pairs = set(); gene_disease_pairs = set(); comp_disease_pairs = set()
for h,r,t in train_int:
    rname = rid_to_relname.get(str(r), str(r))
    h_type = ent2type.get(str(h),"other"); t_type = ent2type.get(str(t),"other")
    rl = rname.lower() if rname else ""
    if h_type=="compound" and t_type=="gene": comp_gene_pairs.add((h,t))
    if h_type=="gene" and t_type=="disease": gene_disease_pairs.add((h,t))
    if h_type=="compound" and t_type=="disease": comp_disease_pairs.add((h,t))
    if relname_indicates(rname, pattern_cg):
        if h_type=="compound" and t_type=="gene": comp_gene_pairs.add((h,t))
        elif t_type=="compound" and h_type=="gene": comp_gene_pairs.add((t,h))
    if relname_indicates(rname, pattern_gd):
        if h_type=="gene" and t_type=="disease": gene_disease_pairs.add((h,t))
        elif t_type=="gene" and h_type=="disease": gene_disease_pairs.add((t,h))
    if "treat" in rl or "indicat" in rl:
        if h_type=="compound" and t_type=="disease": comp_disease_pairs.add((h,t))

print("Mechanistic pairs (train): C->G:",len(comp_gene_pairs),"G->D:",len(gene_disease_pairs),"C->D:",len(comp_disease_pairs))



Mechanistic pairs (train): C->G: 0 G->D: 0 C->D: 0


In [16]:
# -------------------- BUILD COMPOUND-DISEASE PAIRS --------------------
def build_cd_pairs(int_triples):
    pos=[]
    for h,r,t in int_triples:
        h_type = ent2type.get(str(h),"other"); t_type = ent2type.get(str(t),"other")
        if (h_type=="compound" and t_type=="disease") or ("treat" in str(r).lower()) or ("indicat" in str(r).lower()):
            pos.append((h,t))
    return list(set(pos))

train_pos_cd = build_cd_pairs(train_int)
val_pos_cd   = build_cd_pairs(val_int)
test_pos_cd  = build_cd_pairs(test_int)

all_disease_gids = [int(k) for k,v in ent2type.items() if v=="disease"]
all_compound_gids = [int(k) for k,v in ent2type.items() if v=="compound"]

def negative_sampling(pos_list, n_nodes):
    pos_set = set(pos_list)
    neg = []
    while len(neg)<len(pos_list):
        h = random.choice(all_compound_gids)
        t = random.choice(all_disease_gids)
        if (h,t) not in pos_set:
            neg.append((h,t))
    return neg



In [17]:
# -------------------- MODELS --------------------
class GCNModel(nn.Module):
    def __init__(self, num_nodes, hdim=HDIM):
        super().__init__()
        self.emb = nn.Embedding(num_nodes, hdim)
        self.lin = nn.Linear(hdim, hdim)
        self.act = nn.ReLU()
    def forward(self, edge_index, edge_type=None):
        x = self.emb.weight  # full graph embedding
        # naive aggregation: sum of neighbor embeddings
        row, col = edge_index
        agg = torch.zeros_like(x)
        agg.index_add_(0, row, x[col])
        x = self.act(self.lin(agg))
        return x

# DistMult scoring
class DistMult(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.W = nn.Parameter(torch.Tensor(emb_dim))
        nn.init.xavier_uniform_(self.W.data)
    def forward(self, h_emb, t_emb):
        return torch.sum(h_emb * self.W * t_emb, dim=-1)



In [ ]:
# -------------------- TRAIN LOOP --------------------
def train_model(model, edge_index, edge_type, pos_pairs, neg_pairs, epochs=EPOCHS, lr=LR):
    model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=WEIGHT_DECAY)
    criterion = nn.BCEWithLogitsLoss()
    
    for ep in range(epochs):
        model.train()
        all_pairs = pos_pairs + neg_pairs
        random.shuffle(all_pairs)
        labels = torch.tensor([1]*len(pos_pairs)+[0]*len(neg_pairs), dtype=torch.float32, device=DEVICE)
        h_idx = torch.tensor([p[0] for p in all_pairs], device=DEVICE)
        t_idx = torch.tensor([p[1] for p in all_pairs], device=DEVICE)
        opt.zero_grad()
        full_embs = model(edge_index.to(DEVICE), edge_type.to(DEVICE) if edge_type is not None else None)
        h_emb = full_embs[h_idx]
        t_emb = full_embs[t_idx]
        scores = DistMult(HDIM)(h_emb,t_emb)
        loss = criterion(scores, labels)
        loss.backward()
        opt.step()
        if ep%2==0: print(f"Epoch {ep}, Loss: {loss.item():.4f}")
    return model



Pos CD pairs train/val/test: 60568 6545 1668
Train pairs total: 121136 Val pairs: 13090 Test pairs: 3336


In [ ]:
# -------------------- PREDICTION --------------------
def predict(model, edge_index, edge_type, pairs, mc_runs=MC_RUNS):
    model.eval()
    h_idx = torch.tensor([p[0] for p in pairs], device=DEVICE)
    t_idx = torch.tensor([p[1] for p in pairs], device=DEVICE)
    preds=[]
    with torch.no_grad():
        full_embs = model(edge_index.to(DEVICE), edge_type.to(DEVICE) if edge_type is not None else None)
        h_emb = full_embs[h_idx]
        t_emb = full_embs[t_idx]
        scores = torch.sigmoid(DistMult(HDIM)(h_emb,t_emb))
        preds = scores.cpu().numpy()
    return preds



In [ ]:
# -------------------- ENSEMBLE --------------------
trained_models=[]
for i in range(ENSEMBLE_SIZE):
    print(f"Training model {i+1}/{ENSEMBLE_SIZE}")
    neg_pairs = negative_sampling(train_pos_cd, num_nodes)
    model = GCNModel(num_nodes, HDIM)
    model = train_model(model, edge_index, edge_type, train_pos_cd, neg_pairs)
    trained_models.append(model)



NeighborLoader available


In [ ]:
# -------------------- INFERENCE --------------------
all_test_pairs = test_pos_cd + negative_sampling(test_pos_cd, num_nodes)
ensemble_preds = []
for model in trained_models:
    ensemble_preds.append(predict(model, edge_index, edge_type, all_test_pairs))

ensemble_preds = np.mean(np.array(ensemble_preds), axis=0)
h_idx = [p[0] for p in all_test_pairs]
t_idx = [p[1] for p in all_test_pairs]

top_idx = np.argsort(-ensemble_preds)[:TOP_K]
top_pairs = [(h_idx[i], t_idx[i], ensemble_preds[i]) for i in top_idx]




In [ ]:
# -------------------- OUTPUT --------------------
out_df = pd.DataFrame(top_pairs, columns=["compound_gid","disease_gid","score"])
out_df["compound_name"] = out_df["compound_gid"].map(lambda x: ent2name.get(str(x),"UNK"))
out_df["disease_name"] = out_df["disease_gid"].map(lambda x: ent2name.get(str(x),"UNK"))
out_df.to_csv(os.path.join(OUT_DIR,"predictions_top50.csv"), index=False)
print("Top-K predictions saved to CSV")